# ทดสอบจับเลขบัตรประจำตัวประชาชน

In [670]:
import cv2
from matplotlib import table
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

output_folder = Path("../data/output_images/output_V6_TN")
output_folder.mkdir(exist_ok=True)

row_number = 50-2
number_run = 14
#image = Image.open(f"C:/Users/Impan/Documents/ocr-engine-python/data/test_images/transcript/technician/good/transcript_tn_f_{number_run}.png")
image = Image.open(f"C:/Users/Impan/Documents/ocr-engine-python/data/test_images/transcript/technician/slightly_tilted/transcript_tn_st_f_{number_run}.png")
read_sheet_name = f"f_{number_run}"

if image is None:
    raise FileNotFoundError("ไม่พบไฟล์ภาพ กรุณาตรวจสอบเส้นทางของไฟล์")

new_size = (1660, 2347)  # ตัวอย่างขนาดใหม่
resized_pil = image.resize(new_size, Image.LANCZOS) # ปรับขนาดภาพด้วย LANCZOS filter

# แปลงภาพจาก PIL Image เป็น NumPy array (ในรูปแบบ RGB)
img_rgb = np.array(resized_pil)

# แปลงจาก RGB เป็น BGR เพื่อให้ใช้งานกับ OpenCV ได้
img_cv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

denoised = cv2.bilateralFilter(img_cv, d=9, sigmaColor=75, sigmaSpace=75) # จำกัด noise
gray_img = cv2.cvtColor(denoised, cv2.COLOR_BGR2GRAY)

binary_gaussian = cv2.adaptiveThreshold(
    gray_img, 
    maxValue=255, 
    adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    thresholdType=cv2.THRESH_BINARY_INV, 
    blockSize=51, 
    C=15 #21
)

# สร้าง kernel สำหรับ morphological operation
kernel = np.ones((3, 3), np.uint8)
dilated = cv2.dilate(binary_gaussian, kernel, iterations=1)
# ใช้ closing เพื่อเติมเต็มส่วนที่ขาดของเส้น
closed_dummy = cv2.morphologyEx(binary_gaussian, cv2.MORPH_CLOSE, kernel, iterations=1)

cv2.imwrite(f"{output_folder}/img_cv.png", img_cv)
cv2.imwrite(f"{output_folder}/denoised.png", denoised)
cv2.imwrite(f"{output_folder}/dilated.png", dilated)
cv2.imwrite(f"{output_folder}/gray.png", gray_img)
cv2.imwrite(f"{output_folder}/binary_g.png", binary_gaussian)

True

In [671]:
def split_grade_table_and_students(binary_img, denoised, dummy):
    
    # แยกตาราง
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(dummy, connectivity=8)
    areas = [stat[4] for stat in stats]  # ดึงค่า area
    sorted_areas = sorted(areas, reverse=True)  # เรียงลำดับจากมากไปน้อย
    second_max_area = sorted_areas[1]  # ค่าอันดับ 2
    second_max_area_index = areas.index(second_max_area)  # หาตำแหน่งในลิสต์เดิม
    table_position = stats[second_max_area_index]
    x, y, w, h, area = table_position

    table_img = binary_img[y:y+h, x:x+w]
    table_dummy_img = dummy[y:y+h, x:x+w]
    table_original_img = denoised[y:y+h, x:x+w]

    # ข้อมูลนักเรียน
    #x_start = int((x+w) * 0.40) # ความกว้าง 40% ของตาราง
    x_end = int((x+w) * 0.85) # ความกว้าง 85% ของตาราง
    x_split_half = int((x+w) * 0.38) # ความกว้าง 38% ของตาราง

    student_info_img = binary_img[:y, :x_end]
    student_info_fh_img = binary_img[:y, :x_split_half] # ครึ่งแรก
    student_info_sh_img = binary_img[:y, x_split_half:x_end] # ครึ่งหลัง

    return table_img, table_dummy_img, table_original_img, student_info_img, student_info_fh_img, student_info_sh_img

def biggest_contour(contours):
    biggest = np.array([])
    max_area = 0
    for i in contours:
        area = cv2.contourArea(i)
        #print(area)
        if area > 1000:
            #print("มา")
            peri = cv2.arcLength(i, True)
            approx = cv2.approxPolyDP(i, 0.02 * peri, True)
            if area > max_area and len(approx) == 4:
                biggest = approx
                max_area = area

    return biggest

def persective_transformation(table_binary_img, table_original_img, table_dummy_img):

    # ค้นหาคอนทัวร์
    contours, hierarchy = cv2.findContours(table_dummy_img, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    #contours, hierarchy = cv2.findContours(table_binary_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:10]

    # ค้นหาสี่เหลี่ยมที่ใหญ่ที่สุด
    biggest = biggest_contour(contours)

    points = biggest.reshape(4, 2)
    input_points = np.zeros((4, 2), dtype="float32")

    points_sum = points.sum(axis=1)
    input_points[0] = points[np.argmin(points_sum)]
    input_points[3] = points[np.argmax(points_sum)]

    points_diff = np.diff(points, axis=1)
    input_points[1] = points[np.argmin(points_diff)]
    input_points[2] = points[np.argmax(points_diff)]

    (top_left, top_right, bottom_right, bottom_left) = input_points

    # Euclidean Distance Formula
    bottom_width = np.sqrt(((bottom_right[0] - bottom_left[0]) ** 2) + ((bottom_right[1] - bottom_left[1]) ** 2))
    top_width = np.sqrt(((top_right[0] - top_left[0]) ** 2) + ((top_right[1] - top_left[1]) ** 2))
    rigth_height = np.sqrt(((top_left[0] - bottom_right[0]) ** 2) + ((top_left[1] - bottom_right[1]) ** 2))
    left_height = np.sqrt(((top_left[0] - bottom_left[0]) ** 2) + ((top_left[1] - bottom_left[1]) ** 2))

    # Output image size
    #max_width = max(int(bottom_width), int(top_width))
    expand_width = round(max(int(bottom_width), int(top_width)) * 0.4)
    max_width = max(int(bottom_width), int(top_width)) + expand_width
    max_height = max(int(rigth_height), int(left_height))

    # Desird points values in the output image
    converted_points = np.float32([[0, 0], [max_width, 0], [0, max_height], [max_width, max_height]])

    # Perspective transformaxtion
    matrix = cv2.getPerspectiveTransform(input_points, converted_points)
    img_out = cv2.warpPerspective(table_binary_img.copy(), matrix, (max_width, max_height))
    img_original_out = cv2.warpPerspective(table_original_img.copy(), matrix, (max_width, max_height))
    img_dummy_out = cv2.warpPerspective(table_dummy_img.copy(), matrix, (max_width, max_height))

    return img_out, img_original_out, img_dummy_out

table_img, table_dummy_img, table_original_img, student_info_img, student_info_fh_img, student_info_sh_img = split_grade_table_and_students(binary_gaussian, denoised, dilated)
table_persective_img, table_original_persective_img, table_dummy_persective_img = persective_transformation(binary_gaussian, denoised, dilated)

cv2.imwrite(f"{output_folder}/table_img.png", table_img)
cv2.imwrite(f"{output_folder}/table_dummy_img.png", table_dummy_img)
cv2.imwrite(f"{output_folder}/table_original_img.png", table_original_img)
cv2.imwrite(f"{output_folder}/student_info_img.png", student_info_img)
cv2.imwrite(f"{output_folder}/student_info_fh_img.png", student_info_fh_img)
cv2.imwrite(f"{output_folder}/student_info_sh_img.png", student_info_sh_img)

cv2.imwrite(f"{output_folder}/table_persective_img.png", table_persective_img)
cv2.imwrite(f"{output_folder}/table_original_persective_img.png", table_original_persective_img)
cv2.imwrite(f"{output_folder}/table_dummy_persective_img.png", table_dummy_persective_img)


True

In [672]:
def crop_border(image, left_percent=0, right_percent=0, top_percent=0, bottom_percent=0):
    
    # หาความกว้างและความสูงของภาพ
    height, width = image.shape

    # คำนวณพิกัดที่จะตัด (แปลงเป็นพิกเซล)
    x_start = int(width * left_percent)
    x_end = int(width * (1 - right_percent))
    y_start = int(height * top_percent)
    y_end = int(height * (1 - bottom_percent))

    # ตัดภาพ (Crop)
    cropped_img = image[y_start:y_end, x_start:x_end]

    #cv2.imwrite(f"{output_folder}/cropped_fh.jpg", cropped_img)
    
    return cropped_img

def find_text_student_info_fh(student_info_fh_img):
    student_info_fh_img = crop_border(student_info_fh_img.copy(), 0.06, 0.06, 0.06, 0.01)

    rgb_image = cv2.cvtColor(student_info_fh_img.copy(), cv2.COLOR_GRAY2RGB)
    
    # กำหนด kernel (ขนาดของ kernel สามารถปรับเปลี่ยนได้ตามความเหมาะสม)
    kernel_open = np.ones((2, 2), np.uint8)
    kernel_close = np.ones((6, 50), np.uint8)
    
    opening = cv2.morphologyEx(student_info_fh_img.copy(), cv2.MORPH_OPEN, kernel=kernel_open, iterations=1)
    closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel=kernel_close, iterations=2)

    rgb_closing_image = cv2.cvtColor(closing, cv2.COLOR_GRAY2RGB)

    cv2.imwrite(f"{output_folder}/opening.jpg", opening)
    cv2.imwrite(f"{output_folder}/closing.jpg", closing)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(closing, connectivity=8)

    # 1. เอา stats ตัวแรก (background) ออก
    stats_no_bg = stats[1:]

    # 2. เรียง stats ใหม่โดยใช้ค่า area (คอลัมน์ที่ 4) จากมากไปน้อย
    sorted_indices = np.argsort(-stats_no_bg[:, 4])
    sorted_stats = stats_no_bg[sorted_indices]

    # 3. เลือกแค่ 13 element ที่มีค่า area สูงสุด
    top_13_stats = sorted_stats[:13]
    top_13_stats_sorted_by_y = top_13_stats[np.argsort(top_13_stats[:, 1])]

    img_height, img_width = student_info_fh_img.shape[:2]

    # กำหนด margin เป็นเปอร์เซ็นต์ของขนาด bounding box
    # เช่น กำหนด 10% ของความกว้าง/ความสูงของ bounding box สำหรับแต่ละด้าน
    left_margin_percent = 0.1    # ขยายซ้าย 10%
    right_margin_percent = 0.1   # ขยายขวา 10%
    top_margin_percent = 0.2     # ขยายบน 20%
    bottom_margin_percent = 0.1  # ขยายล่าง 10%

    text_group_stud_fh = []
    for idx, stats in enumerate(top_13_stats_sorted_by_y): # เก็บภาพกลุม
        x, y, w, h, area = stats

        # คำนวณ margin ตามเปอร์เซ็นต์ของ bounding box
        left_margin = int(w * left_margin_percent)
        right_margin = int(w * right_margin_percent)
        top_margin = int(h * top_margin_percent)
        bottom_margin = int(h * bottom_margin_percent)

        # คำนวณพิกัดใหม่โดยใช้ margin ที่คำนวณได้
        x_new = max(x - left_margin, 0)
        y_new = max(y - top_margin, 0)
        x_end = min(x + w + right_margin, img_width)
        y_end = min(y + h + bottom_margin, img_height)

        cluster_img = student_info_fh_img[y_new:y_end, x_new:x_end]
        text_group_stud_fh.append(cluster_img)

        # วาดกรอบที่ขยายแล้วลงบนภาพ
        cv2.rectangle(rgb_image, (x_new, y_new), (x_end, y_end), (0, 255, 0), 1)
        cv2.rectangle(rgb_closing_image, (x_new, y_new), (x_end, y_end), (0, 255, 0), 1)
        
    cv2.imwrite(f"{output_folder}/cca_top_13_stats.jpg", rgb_image)
    cv2.imwrite(f"{output_folder}/cca_rgb_closing_image.jpg", rgb_closing_image)

    return text_group_stud_fh[1:]

def find_text_student_info_sh(student_info_sh_img):
    student_info_sh_img = crop_border(student_info_sh_img.copy(), 0.05, 0.00, 0.05, 0.01)

    rgb_image = cv2.cvtColor(student_info_sh_img.copy(), cv2.COLOR_GRAY2RGB)
    
    # กำหนด kernel (ขนาดของ kernel สามารถปรับเปลี่ยนได้ตามความเหมาะสม)
    kernel_open = np.ones((2, 2), np.uint8)
    kernel_close = np.ones((8, 50), np.uint8)
    
    opening = cv2.morphologyEx(student_info_sh_img.copy(), cv2.MORPH_OPEN, kernel=kernel_open, iterations=1)
    closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel=kernel_close, iterations=2)

    rgb_closing_image = cv2.cvtColor(closing, cv2.COLOR_GRAY2RGB)

    cv2.imwrite(f"{output_folder}/opening_sh.jpg", opening)
    cv2.imwrite(f"{output_folder}/closing_sh.jpg", closing)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(closing, connectivity=8)

    # 1. เอา stats ตัวแรก (background) ออก
    stats_no_bg = stats[1:]

    # 2. เรียง stats ใหม่โดยใช้ค่า area (คอลัมน์ที่ 4) จากมากไปน้อย
    sorted_indices = np.argsort(-stats_no_bg[:, 4])
    sorted_stats = stats_no_bg[sorted_indices]

    # 3. เลือกแค่ 14 element ที่มีค่า area สูงสุด
    top_14_stats = sorted_stats[:14]
    top_14_stats_sorted_by_y = top_14_stats[np.argsort(top_14_stats[:, 1])]

    img_height, img_width = student_info_sh_img.shape[:2]

    # กำหนด margin เป็นเปอร์เซ็นต์ของขนาด bounding box
    # เช่น กำหนด 10% ของความกว้าง/ความสูงของ bounding box สำหรับแต่ละด้าน
    left_margin_percent = 0.1    # ขยายซ้าย 10%
    right_margin_percent = 0.1   # ขยายขวา 10%
    top_margin_percent = 0.2     # ขยายบน 20%
    bottom_margin_percent = 0.1  # ขยายล่าง 10%

    text_group_stud_sh = []
    for idx, stats in enumerate(top_14_stats_sorted_by_y): # เก็บภาพกลุม
        x, y, w, h, area = stats

        # คำนวณ margin ตามเปอร์เซ็นต์ของ bounding box
        left_margin = int(w * left_margin_percent)
        right_margin = int(w * right_margin_percent)
        top_margin = int(h * top_margin_percent)
        bottom_margin = int(h * bottom_margin_percent)

        # คำนวณพิกัดใหม่โดยใช้ margin ที่คำนวณได้
        x_new = max(x - left_margin, 0)
        y_new = max(y - top_margin, 0)
        x_end = min(x + w + right_margin, img_width)
        y_end = min(y + h + bottom_margin, img_height)

        cluster_img = student_info_sh_img[y_new:y_end, x_new:x_end]
        text_group_stud_sh.append(cluster_img)

        # วาดกรอบที่ขยายแล้วลงบนภาพ
        cv2.rectangle(rgb_image, (x_new, y_new), (x_end, y_end), (0, 255, 0), 1)
        cv2.rectangle(rgb_closing_image, (x_new, y_new), (x_end, y_end), (0, 255, 0), 1)
        
    cv2.imwrite(f"{output_folder}/cca_top_14_stats.jpg", rgb_image)
    cv2.imwrite(f"{output_folder}/cca_rgb_closing_image.jpg", rgb_closing_image)

    return text_group_stud_sh[3:]

text_stud_fh_images = find_text_student_info_fh(student_info_fh_img)
text_stud_sh_images = find_text_student_info_sh(student_info_sh_img)

In [673]:
indices_fh = [3, -2, -1, 6]
indices_sh = [-3, -1]
student_name, field_of_study, field_of_work, citizen_id = [text_stud_fh_images[i] for i in indices_fh]
gpa, graduation_date = [text_stud_sh_images[i] for i in indices_sh]

In [674]:
def detect_sub_text_in_group_stud(binary_image, mode=0):
    debug = False

    if debug == True:
        plt.figure(figsize=(5,5))
        plt.imshow(binary_image, cmap="gray")
        plt.title(f"binary_image")
        plt.show()
    
    text_group = []

    '''
    kernel_open = np.ones((2, 2), np.uint8)
    remove_noise = cv2.morphologyEx(binary_image, cv2.MORPH_OPEN, kernel_open, iterations=1)

    if debug == True:
        plt.figure(figsize=(5,5))
        plt.imshow(remove_noise, cmap="gray")
        plt.title(f"remove_noise")
        plt.show()'
    '''

    if (mode == 1):
        kernel = np.ones((5, 16), np.uint8)
        dummy_image = cv2.dilate(binary_image, kernel, iterations=1)
        #dummy_image = cv2.morphologyEx(binary_image, cv2.MORPH_CLOSE, kernel, iterations=1)
    elif (mode == 2):
        kernel = np.ones((5, 6), np.uint8)
        dummy_image = cv2.dilate(binary_image, kernel, iterations=1)
        #dummy_image = cv2.morphologyEx(binary_image, cv2.MORPH_CLOSE, kernel, iterations=1)
    else:
        kernel = np.ones((5, 6), np.uint8)
        dummy_image = cv2.dilate(binary_image, kernel, iterations=1)
        #dummy_image = cv2.morphologyEx(binary_image, cv2.MORPH_CLOSE, kernel, iterations=1)

    
    if debug == True:
        print("Mode:",mode)
        plt.figure(figsize=(5,5))
        plt.imshow(dummy_image, cmap="gray")
        plt.title(f"dummy_image")
        plt.show()
        

    # ใช้ Connected Component Analysis
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(dummy_image, connectivity=8)
    word_stats = stats[1:] # ข้าม Background (index 0)
    sorted_indices = np.argsort(word_stats[:, 0]) # จัดเรียงตามค่า x (คอลัมน์ที่ 0)
    sorted_stats = word_stats[sorted_indices]

    for idx, stats in enumerate(sorted_stats):
        x, y, w, h, area = stats
        cluster_img = binary_image[y:y+h, x:x+w]
        text_group.append(cluster_img)

    return text_group

text_group_student_name = detect_sub_text_in_group_stud(student_name)
text_group_field_of_study = detect_sub_text_in_group_stud(field_of_study)
text_group_field_of_work = detect_sub_text_in_group_stud(field_of_work)
text_group_citizen_id = detect_sub_text_in_group_stud(citizen_id, mode=1)

text_group_gpa = detect_sub_text_in_group_stud(gpa, mode=2)
text_group_graduation_date = detect_sub_text_in_group_stud(graduation_date)

In [675]:
def detect_one_level_of_char_stud(text_group):
    debug = False
    char_images = []

    for idx_s, sub_text in enumerate(text_group):

        if debug == True:
            plt.figure(figsize=(3, 3))
            plt.imshow(sub_text, cmap="gray")
            plt.title(f"sub text:{idx_s+1}")
            plt.show()
            
        #skeleton = cv2.ximgproc.thinning(sub_text, thinningType=cv2.ximgproc.THINNING_ZHANGSUEN)
        skeleton_guohall = cv2.ximgproc.thinning(sub_text, thinningType=cv2.ximgproc.THINNING_GUOHALL)

        if debug == True:
            plt.figure(figsize=(3, 3))
            plt.imshow(skeleton_guohall, cmap="gray")
            plt.title(f"skeleton, sub text:{idx_s+1}")
            plt.show()
            
            
        #kernel_open = np.ones((2, 2), np.uint8)
        kernel_dummy = np.ones((2, 2), np.uint8)
        #opening = cv2.morphologyEx(skeleton, cv2.MORPH_OPEN, kernel=kernel_open, iterations=2)
        #closing = cv2.morphologyEx(skeleton, cv2.MORPH_CLOSE, kernel=kernel_open, iterations=2)
        dummy_image = cv2.dilate(skeleton_guohall, kernel_dummy, iterations=1)

        if debug == True:
            plt.figure(figsize=(3, 3))
            plt.imshow(dummy_image, cmap="gray")
            plt.title(f"dummy_image, sub text:{idx_s+1}")
            plt.show()
            
            
        rgb_image = cv2.cvtColor(sub_text.copy(), cv2.COLOR_GRAY2RGB)

        contours, hierarchy = cv2.findContours(dummy_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        sorted_contours = sorted(contours, key=lambda cnt: cv2.boundingRect(cnt)[0])

        
        for idx_c, cnt in enumerate(sorted_contours):

            x, y, w, h = cv2.boundingRect(cnt)
            contour_area = cv2.contourArea(cnt)

            mask = np.zeros(sub_text.shape[:2], dtype=np.uint8)
            cv2.drawContours(mask, [cnt], -1, 255, -1)

            kernel_mask = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))  # ปรับขนาด kernel ตามต้องการ
            dilated_mask = cv2.dilate(mask, kernel_mask, iterations=1)

            # ใช้ mask กับภาพต้นฉบับ เพื่อดึงเฉพาะส่วนภายใน contour
            char_result = cv2.bitwise_and(sub_text, sub_text, mask=dilated_mask)

            contours_char, hierarchy_char = cv2.findContours(char_result, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            largest_contour = max(contours_char, key=cv2.contourArea)

            x, y, w, h = cv2.boundingRect(largest_contour)
            area = int(cv2.contourArea(largest_contour))
            crop_img = char_result[y:y+h, x:x+w]
            char_images.append(crop_img)
                
            if debug == True:
                plt.figure(figsize=(2, 2))
                plt.imshow(crop_img, cmap="gray")
                plt.title(f"crop_img, sub text:{idx_s+1}, char:{idx_c+1}")
                plt.show()
            #print(f"Contour #{idx_c}: bounding box = (x={x}, y={y}, w={w}, h={h}, area={contour_area})")
    return char_images

text_group_char_citizen_id = detect_one_level_of_char_stud(text_group_citizen_id[1:])





In [676]:
from tensorflow.keras.models import load_model

model_path_char_subject_code_tn = "../models/char_subject_code_tn_model.h5"
model_path_char_academic_results_tn = "../models/char_academic_results_tn_model.h5"

model_char_subject_code_tn = load_model(model_path_char_subject_code_tn)
model_char_academic_results_tn= load_model(model_path_char_academic_results_tn)

# สร้าง Mapping ของโมเดลตามระดับ
models_one_level = {
    0: model_char_subject_code_tn,
    1: model_char_academic_results_tn,
}

In [677]:
char_subject_code_tn = [
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '-',
]

char_academic_results_tn = [
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
    'ก', 'ข', 'ถ', 'ท', 'น', 'ป', 'ผ', 'ม', 'ร', 'ล', 'ส', '.'
]

char_labels = {
    0: char_subject_code_tn,
    1: char_academic_results_tn,
}

def resize_with_min_padding(image, desired_size, min_padding):
    
    """
    ปรับขนาดภาพให้ใกล้เคียง desired_size โดยลด Padding และเพิ่มการขยายภาพต้นฉบับ
    """
    if image is None or not isinstance(image, np.ndarray):
        raise ValueError("Input image must be a valid numpy array.")

    if not isinstance(desired_size, int) or desired_size <= 0:
        raise ValueError("desired_size must be a positive integer.")

    old_size = image.shape[:2]  # (height, width)
    max_size = max(old_size)

    # คำนวณอัตราส่วนการปรับขนาดให้ใกล้เคียง desired_size
    ratio = float(desired_size - 2 * min_padding) / max_size
    new_size = tuple([int(x * ratio) for x in old_size])  # ขนาดใหม่ (height, width)

    # Resize ภาพให้คงสัดส่วนเดิม แต่ใหญ่ขึ้น
    resized_image = cv2.resize(image, (new_size[1], new_size[0]), interpolation=cv2.INTER_AREA)

    # คำนวณ Padding ใหม่
    delta_w = max(desired_size - new_size[1], 0)  # Padding ด้านความกว้าง
    delta_h = max(desired_size - new_size[0], 0)  # Padding ด้านความสูง
    top, bottom = delta_h // 2, delta_h - (delta_h // 2)
    left, right = delta_w // 2, delta_w - (delta_w // 2)

    # ตรวจสอบสีสำหรับ Grayscale หรือ RGB
    color = [0] if len(image.shape) == 2 else [0, 0, 0]

    # เพิ่ม Padding รอบภาพ
    padded_image = cv2.copyMakeBorder(resized_image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)

    return padded_image

def predict_text_one_level_stud(text_group_char, char_model=0):
    # กำหนดขนาด Input ของโมเดล
    input_size = 32  # ขนาด 32x32
    text_block = ""

    for idx_c, char in enumerate(text_group_char):

        #plt.figure(figsize=(2, 2))
        #plt.imshow(char, cmap="gray")
        #plt.title(f"char, sub text:{idx_s+1}, char:{idx_c+1}")
        #plt.show()
                
        if char is None:
            print(f"Character image {idx_c} is None.")
            continue  # ข้ามภาพนี้
        else:
            # เพิ่ม Padding และปรับขนาดภาพ
            padded_img = resize_with_min_padding(char, input_size, min_padding=1)

            '''
            plt.figure(figsize=(2, 2))
            plt.imshow(padded_img, cmap="gray")
            plt.title(f"char_resize, text box:{idx_g+1}, sub text:{idx_s+1}, char:{idx_c+1}")
            plt.show()
            ''' 
                
            # Normalization (เปลี่ยนค่าพิกเซลให้อยู่ในช่วง [0, 1])
            normalized_img = padded_img / 255.0

            if len(normalized_img.shape) == 2:  # หากภาพเป็น Grayscale (2D)
                normalized_img = np.expand_dims(normalized_img, axis=-1)
                processed_image = np.expand_dims(normalized_img, axis=0)  # เพิ่ม Batch Dimension

            if char_model in models_one_level:
                prediction = models_one_level[char_model].predict(processed_image)
                predicted_class = np.argmax(prediction)
                confidence_score = np.max(prediction)

                class_char = char_labels[char_model]
                predicted_letter = class_char[predicted_class]
                text_block += predicted_letter
            '''
            if len(normalized_img.shape) == 2:  # หากภาพเป็น Grayscale (2D)
                normalized_img = np.expand_dims(normalized_img, axis=-1)
                processed_image = np.expand_dims(normalized_img, axis=0)  # เพิ่ม Batch Dimension

                prediction = model_char_subject_code_tn.predict(processed_image)
                predicted_class = np.argmax(prediction)

                char_label_model = char_labels[label]
                predicted_letter = char_label_model[predicted_class]

                sub_text_result += predicted_letter
            '''
        
    print("ประมวลผลเสร็จสิ้น")
    return text_block

text_box_citizen_id = predict_text_one_level_stud(text_group_char_citizen_id[:], 0)   



1/1 [==============================] - 0s 14ms/step
ประมวลผลเสร็จสิ้น


In [678]:
print(text_box_citizen_id.replace(" ", ""))

1570501300324


In [679]:
import pandas as pd

# อ่านไฟล์ Excel (กำหนด sheet name หากต้องการ เช่น 'Sheet1')
#excel_file = "C:/Users/Impan/Documents/ocr-engine-python/data/excel/ts_tn_g.xlsx"
excel_file = "C:/Users/Impan/Documents/ocr-engine-python/data/excel/ts_tn_st.xlsx"
df = pd.read_excel(excel_file, sheet_name=read_sheet_name, header=None)

# ดึงข้อมูลจากเซลล์ B1 ถึง B6 (index เริ่มต้นที่ 0)
exl_file_name       = df.iloc[0, 1]
exl_name            = df.iloc[1, 1]
exl_citizen_id      = df.iloc[2, 1]
exl_graduation_date = df.iloc[3, 1]
exl_field_of_study  = df.iloc[4, 1]
exl_field_of_work   = df.iloc[5, 1]
exl_cgpa            = df.iloc[6, 1]


# แสดงผลตัวแปรที่ได้
print("File Name:", exl_file_name)
print("Name:", exl_name)
print("Citizen ID:", exl_citizen_id)
print("Graduation Date:", exl_graduation_date)
print("Field of Study:", exl_field_of_study)
print("Field of Work:", exl_field_of_work)
print("CGPA:", exl_cgpa)

print("ORC Citizen ID:",text_box_citizen_id)

File Name: transcript_tn_st_f_14
Name: นายสิรดนัย กำวิน
Citizen ID: 1 5705 01304 32 4
Graduation Date: 2566-03-28 00:00:00
Field of Study: โยธา
Field of Work: โยธา
CGPA: 3.5
ORC Citizen ID: 1570501300324


In [680]:
import Levenshtein

def compare_text(text1, text2):
    debug = False

    distance = Levenshtein.distance(text1, text2)
    ratio = Levenshtein.ratio(text1, text2)  # อยู่ในช่วง 0.0 - 1.0
    percent_similarity = ratio * 100

    if(debug == True):
        print(f"Distance: {distance}")
        print(f"Similarity: {percent_similarity:.2f}%")

    return percent_similarity

result_citizen_id = compare_text(exl_citizen_id.replace(" ", ""), text_box_citizen_id.replace(" ", ""))

print(f"เลขบัตรประจำตัว:{result_citizen_id:.2f}%")


เลขบัตรประจำตัว:92.31%


In [681]:
import pandas as pd

# ตั้งค่าชื่อไฟล์และ sheet ที่ต้องการ
excel_file = 'C:/Users/Impan/Documents/ocr-engine-python/data/excel/experimental_results.xlsx'
sheet_name = 'tn_we'

# อ่านข้อมูลจากไฟล์ Excel เข้ามาเป็น DataFrame
df = pd.read_excel(excel_file, sheet_name=sheet_name)

# เปลี่ยนค่าในเซลล์ B2 (แถวที่ 2, คอลัมน์ที่ 3) โดยใช้ iloc (index เริ่มที่ 0)
df.iloc[row_number, 2] = result_citizen_id  # เปลี่ยนค่าเป็น

# บันทึกการแก้ไขกลับไปที่ไฟล์ Excel
with pd.ExcelWriter(excel_file, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
    df.to_excel(writer, sheet_name=sheet_name, index=False)
